# Week 2 — Neural Networks Lab

Andrew Ng Machine Learning Specialisation — Course 1, Week 2

本 Notebook 示範如何使用 TensorFlow/Keras 建立簡單神經網路，用於多感測器設備狀態分類。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print('Libraries loaded successfully.')

## 1. 產生多感測器資料 / Simulate Multi-Sensor Data

模擬工業場景：振動 + 溫度 + 電流 → 設備健康狀態（0=正常, 1=異常）

In [ ]:
np.random.seed(42)
n_samples = 500

# 正常樣本 (class 0)
normal = np.column_stack([
    np.random.normal(2.0, 0.5, n_samples // 2),   # vibration (mm/s)
    np.random.normal(60.0, 5.0, n_samples // 2),  # temperature (°C)
    np.random.normal(5.0, 0.3, n_samples // 2),   # current (A)
])

# 異常樣本 (class 1)
anomaly = np.column_stack([
    np.random.normal(8.0, 1.5, n_samples // 2),   # vibration (mm/s)
    np.random.normal(90.0, 8.0, n_samples // 2),  # temperature (°C)
    np.random.normal(7.5, 0.8, n_samples // 2),   # current (A)
])

X = np.vstack([normal, anomaly])
y = np.array([0] * (n_samples // 2) + [1] * (n_samples // 2))

print(f'Dataset shape: X={X.shape}, y={y.shape}')
print(f'Class distribution: Normal={np.sum(y==0)}, Anomaly={np.sum(y==1)}')

## 2. 資料前處理 / Preprocessing

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Train: {X_train_scaled.shape}, Test: {X_test_scaled.shape}')

## 3. 建立神經網路 / Build Neural Network

> **Note**: 若尚未安裝 TensorFlow，請先執行 `pip install tensorflow`
> 或使用 scikit-learn MLPClassifier 替代（已在下一格提供）

In [ ]:
# --- 方案 A: scikit-learn MLP (無需 TensorFlow) ---
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report

mlp = MLPClassifier(
    hidden_layer_sizes=(16, 8),
    activation='relu',
    max_iter=500,
    random_state=42,
)
mlp.fit(X_train_scaled, y_train)

y_pred = mlp.predict(X_test_scaled)
print(classification_report(y_test, y_pred, target_names=['Normal', 'Anomaly']))

## 4. 視覺化決策邊界 / Visualise (2D projection)

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(
    X_test_scaled[y_test == 0, 0],
    X_test_scaled[y_test == 0, 1],
    label='Normal', alpha=0.6
)
plt.scatter(
    X_test_scaled[y_test == 1, 0],
    X_test_scaled[y_test == 1, 1],
    label='Anomaly', alpha=0.6, marker='x'
)
plt.xlabel('Vibration (scaled)')
plt.ylabel('Temperature (scaled)')
plt.title('Multi-Sensor Anomaly Detection')
plt.legend()
plt.grid(True)
plt.show()